# DeepSeek Agent 实战：小红书爆款文案生成助手

本 Notebook 将指导您如何使用 DeepSeek LLM 构建一个能够生成小红书爆款文案的智能 Agent。我们将从需求拆解开始，逐步定义 Agent 的系统提示词 (System Prompt)、外部工具 (Tools)，并实现其核心的工作流程，最终生成符合小红书平台特点的文案。

## 1. 环境准备与DeepSeek API配置

In [1]:
import os
from openai import OpenAI

# 建议将 API Key 设置为环境变量，避免直接暴露在代码中
# 从环境变量获取 DeepSeek API Key
api_key = "sk-d973cb458c1141e2bffc9e8988f74656"  # os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    raise ValueError("请设置 DEEPSEEK_API_KEY 环境变量")

# 初始化 DeepSeek 客户端

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

## 2. 需求拆解与Agent任务规划

#### 用户痛点与核心需求：
*   **效率低下：** 人工创作周期长，难以满足高频发布需求。
*   **创意瓶颈：** 难以持续产出新颖、吸引人的爆款创意。
*   **趋势捕捉难：** 实时流行元素难以快速融入文案。
*   **平台特性把握：** 小红书特有风格（标题、正文、标签、表情）难以精准复制。

#### “爆款”文案的特征：
*   **强吸引力标题：** 制造好奇、痛点共鸣、利益点清晰。
*   **沉浸式正文：** 真实体验分享、细节描述、情感共鸣。
*   **精准且多样标签：** 热门话题、品牌词、产品词、垂直领域词。
*   **生动表情符号：** 增强表达力，提升活泼感。
*   **清晰的行动召唤 (CTA)。**

#### Agent 任务规划：核心工作流
1.  **用户指令接收：** 接收产品信息、主题、风格等。
2.  **信息收集 (Web Search/DB Query)：** 实时搜索行业趋势、热门话题、竞品分析、产品卖点。
3.  **内容构思与初稿生成 (LLM)：** 结合所有信息，撰写标题、正文、标签、表情符号。
4.  **风格与格式优化 (LLM)：** 根据小红书平台特点和指定风格，对文案进行润色和结构调整。
5.  **最终输出：** 呈现完整文案。

## 3. 爆款文案生成逻辑与 Prompt 设计

### 3.1 System Prompt (系统提示词)

System Prompt 是 Agent 的“大脑”和“行为准则”。它定义了 Agent 的角色、目标以及工作方式。我们将采用 `Thought-Action-Observation` (ReAct) 模式来引导 DeepSeek 的推理过程。

In [2]:
SYSTEM_PROMPT = """
你是一个资深的小红书爆款文案专家，擅长结合最新潮流和产品卖点，创作引人入胜、高互动、高转化的笔记文案。

你的任务是根据用户提供的产品和需求，生成包含标题、正文、相关标签和表情符号的完整小红书笔记。

请始终采用'Thought-Action-Observation'模式进行推理和行动。文案风格需活泼、真诚、富有感染力。当完成任务后，请以JSON格式直接输出最终文案，格式如下：
```json
{
  "title": "小红书标题",
  "body": "小红书正文",
  "hashtags": ["#标签1", "#标签2", "#标签3", "#标签4", "#标签5"],
  "emojis": ["✨", "🔥", "💖"]
}
```
在生成文案前，请务必先思考并收集足够的信息。
"""

### 3.2 Tools (工具定义)

Agent 的"双手"由一系列可调用的工具组成。这些工具扩展了 LLM 的能力，使其能够获取实时信息、查询数据库或执行特定操作。在这里，我们定义了三个核心工具：

*   `search_web`: 用于搜索互联网上的实时信息，如最新趋势、用户评价等。
*   `query_product_database`: 用于查询产品数据库，获取产品的详细卖点、成分、适用人群、使用方法等信息。基于本地真实产品数据（25款热门护肤品），支持模糊搜索和关键词匹配。
*   `generate_emoji`: 用于根据文案内容生成恰当的表情符号。

In [3]:
TOOLS_DEFINITION = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网上的实时信息，用于获取最新新闻、流行趋势、用户评价、行业报告等。请确保搜索关键词精确，避免宽泛的查询。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "要搜索的关键词或问题，例如'最新小红书美妆趋势'或'深海蓝藻保湿面膜 用户评价'"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "query_product_database",
            "description": "查询内部产品数据库，获取指定产品的详细卖点、成分、适用人群、使用方法等信息。",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {
                        "type": "string",
                        "description": "要查询的产品名称，例如'深海蓝藻保湿面膜'"
                    }
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_emoji",
            "description": "根据提供的文本内容，生成一组适合小红书风格的表情符号。",
            "parameters": {
                "type": "object",
                "properties": {
                    "context": {
                        "type": "string",
                        "description": "文案的关键内容或情感，例如'惊喜效果'、'补水保湿'"
                    }
                },
                "required": ["context"]
            }
        }
    }
]

### 3.3 工具实现

本节实现了三个工具函数：

*   `search_web`: 模拟网页搜索（预留接口，可替换为真实搜索API）。
*   `query_product_database`: **真实产品数据查询**，基于本地 `products_data.json` 文件（包含25款热门护肤品的详细信息），支持精确匹配和模糊搜索（按产品名称、品牌、分类、标签等多维度匹配）。
*   `generate_emoji`: 根据上下文关键词生成适合小红书风格的表情符号。

In [4]:
import json
import random
import os
from difflib import SequenceMatcher

# ============================================================
# 加载真实产品数据
# ============================================================
# 获取产品数据文件路径（与 notebook 同目录）
_CURRENT_DIR = os.path.dirname(os.path.abspath("rednote.ipynb")) if os.path.exists("rednote.ipynb") else os.getcwd()
_PRODUCTS_DATA_PATH = os.path.join(_CURRENT_DIR, "products_data.json")

def _load_products_data():
    """加载本地产品数据文件。"""
    try:
        with open(_PRODUCTS_DATA_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"[数据加载] 成功加载 {len(data['products'])} 条产品数据")
        return data["products"]
    except FileNotFoundError:
        print(f"[警告] 未找到产品数据文件: {_PRODUCTS_DATA_PATH}")
        return []
    except json.JSONDecodeError as e:
        print(f"[错误] 产品数据文件格式有误: {e}")
        return []

# 模块级缓存，避免重复加载
_PRODUCTS_DB = _load_products_data()

# ============================================================
# 工具1: search_web（模拟网页搜索）
# ============================================================
def mock_search_web(query: str) -> str:
    """模拟网页搜索工具，返回预设的搜索结果。"""
    print(f"[Tool Call] 搜索网页：{query}")
    # 此处可替换为真实搜索API，如 Google Search API、SerpAPI 等
    if "小红书美妆趋势" in query:
        return "近期小红书美妆流行'多巴胺穿搭'、'早C晚A'护肤理念、'伪素颜'妆容，热门关键词有#氛围感、#抗老、#屏障修复。"
    elif "保湿面膜" in query:
        return "小红书保湿面膜热门话题：沙漠干皮救星、熬夜急救面膜、水光肌养成。用户痛点：卡粉、泛红、紧绷感。"
    elif "美白" in query or "提亮" in query:
        return "小红书美白精华热门话题：冷白皮养成、烟酰胺精华测评、早C晚A搭配。用户关注：温和不刺激、淡痘印效果。"
    elif "敏感肌" in query or "修护" in query:
        return "小红书敏感肌修护热门话题：屏障修复、泛红急救、温和成分党。热门成分：神经酰胺、积雪草、泛醇。"
    elif "抗老" in query or "紧致" in query:
        return "小红书抗老热门话题：玻色因面霜测评、A醇入门、胶原蛋白精华。用户关注：淡化细纹、紧致轮廓。"
    elif "祛痘" in query or "控油" in query:
        return "小红书祛痘热门话题：刷酸入门、水杨酸精华、油皮护肤。用户关注：温和去黑头、闭口克星、控油不拔干。"
    else:
        return f"未找到关于 '{query}' 的特定信息，但市场反馈通常关注产品成分、功效和用户体验。"

# ============================================================
# 工具2: query_product_database（真实产品数据查询）
# ============================================================
def query_product_database(product_name: str) -> str:
    """
    查询真实产品数据库，支持精确匹配和模糊搜索。
    
    搜索策略：
    1. 精确匹配产品名称
    2. 模糊匹配：基于名称相似度
    3. 关键词匹配：在品牌、分类、标签、成分中搜索
    """
    print(f"[Tool Call] 查询产品数据库：{product_name}")
    
    if not _PRODUCTS_DB:
        return f"产品数据库为空，请检查数据文件是否正确加载。"
    
    # --- 策略1: 精确匹配 ---
    for product in _PRODUCTS_DB:
        if product["name"] == product_name:
            return _format_product_info(product)
    
    # --- 策略2: 名称包含匹配 ---
    matched_by_contains = []
    for product in _PRODUCTS_DB:
        if product_name in product["name"] or product["name"] in product_name:
            matched_by_contains.append(product)
    if matched_by_contains:
        if len(matched_by_contains) == 1:
            return _format_product_info(matched_by_contains[0])
        else:
            results = []
            for p in matched_by_contains[:3]:
                results.append(_format_product_info(p))
            return f"找到 {len(matched_by_contains)} 个相关产品：\n\n" + "\n---\n".join(results)
    
    # --- 策略3: 多维度关键词匹配 ---
    keyword_matches = []
    keywords = product_name.lower().split()
    for product in _PRODUCTS_DB:
        score = 0
        search_fields = [
            product["name"], 
            product["brand"], 
            product["category"],
            " ".join(product["core_ingredients"]),
            " ".join(product["tags"]),
            product["main_benefits"],
            product["suitable_for"]
        ]
        combined_text = " ".join(search_fields).lower()
        for kw in keywords:
            if kw in combined_text:
                score += 1
        name_similarity = SequenceMatcher(None, product_name.lower(), product["name"].lower()).ratio()
        score += name_similarity * 2
        if score > 0.5:
            keyword_matches.append((score, product))
    
    if keyword_matches:
        keyword_matches.sort(key=lambda x: x[0], reverse=True)
        best_match = keyword_matches[0][1]
        if len(keyword_matches) == 1:
            return _format_product_info(best_match)
        else:
            results = []
            for score, p in keyword_matches[:3]:
                results.append(_format_product_info(p))
            return f"找到 {len(keyword_matches)} 个相关产品：\n\n" + "\n---\n".join(results)
    
    return f"产品数据库中未找到与 '{product_name}' 匹配的产品。数据库中共有 {len(_PRODUCTS_DB)} 款产品，可尝试搜索：精华、面霜、面膜、美白、保湿、抗老、祛痘、敏感肌等关键词。"

def _format_product_info(product: dict) -> str:
    """格式化产品信息为可读文本。"""
    return (
        f"产品名称：{product['name']}\n"
        f"品牌：{product['brand']}\n"
        f"分类：{product['category']}\n"
        f"价格：{product['price']}\n"
        f"核心成分：{'、'.join(product['core_ingredients'])}\n"
        f"主要功效：{product['main_benefits']}\n"
        f"适用人群：{product['suitable_for']}\n"
        f"质地：{product['texture']}\n"
        f"使用方法：{product['usage']}\n"
        f"核心卖点：{product['key_selling_points']}\n"
        f"规格：{product['spec']}\n"
        f"评分：{product['rating']}/5.0\n"
        f"标签：{'、'.join(product['tags'])}"
    )

# ============================================================
# 工具3: generate_emoji（表情符号生成）
# ============================================================
# 小红书风格表情符号库，按场景分类
EMOJI_MAP = {
    "补水": ["💦", "💧", "🌊", "✨"],
    "保湿": ["💦", "💧", "🌊", "✨"],
    "水润": ["💦", "💧", "🌊", "✨"],
    "惊喜": ["💖", "😍", "🤩", "💯"],
    "哇塞": ["💖", "😍", "🤩", "💯"],
    "爱了": ["💖", "😍", "🤩", "💯"],
    "推荐": ["✅", "👍", "⭐", "🛍️"],
    "好物": ["✅", "👍", "⭐", "🛍️"],
    "熬夜": ["😭", "😮‍💨", "😴", "💡"],
    "疲惫": ["😭", "😮‍💨", "😴", "💡"],
    "美白": ["✨", "🌟", "💫", "🤍"],
    "提亮": ["✨", "🌟", "💫", "🤍"],
    "修护": ["🌿", "🛡️", "💚", "💪"],
    "敏感": ["🌿", "🛡️", "💚", "💪"],
    "抗老": ["👑", "💎", "⏰", "🔄"],
    "紧致": ["👑", "💎", "⏰", "🔄"],
    "祛痘": ["🎯", "⚡", "🔥", "💪"],
    "控油": ["🎯", "⚡", "🔥", "💪"],
    "防晒": ["☀️", "🌤️", "🧴", "⛱️"],
    "晒后": ["🧊", "❄️", "🌿", "💧"],
}

def generate_emoji(context: str) -> list:
    """根据上下文关键词生成适合小红书风格的表情符号。"""
    print(f"[Tool Call] 生成表情符号，上下文：{context}")
    emojis = []
    for keyword, emoji_list in EMOJI_MAP.items():
        if keyword in context:
            emojis.extend(emoji_list)
    # 去重并限制数量
    emojis = list(dict.fromkeys(emojis))[:8]
    if not emojis:
        emojis = random.sample(["✨", "🔥", "💖", "💯", "🎉", "👍", "🤩", "💧", "🌿"], k=5)
    return emojis

# ============================================================
# 工具注册表
# ============================================================
available_tools = {
    "search_web": mock_search_web,
    "query_product_database": query_product_database,
    "generate_emoji": generate_emoji,
}

[数据加载] 成功加载 25 条产品数据


## 4. 实战：构建小红书文案生成 Agent

现在，我们将把 System Prompt、工具定义和工具函数整合起来，构建出能够自动执行的 DeepSeek Agent 工作流。核心是 `generate_rednote` 函数，它通过一个循环来实现 Agent 的 `Thought-Action-Observation` 过程。其中 `query_product_database` 已接入真实产品数据。

In [5]:
import json
import re

def generate_rednote(product_name: str, tone_style: str = "活泼甜美", max_iterations: int = 5) -> str:
    """
    使用 DeepSeek Agent 生成小红书爆款文案。
    
    Args:
        product_name (str): 要生成文案的产品名称。
        tone_style (str): 文案的语气和风格，如"活泼甜美"、"知性"、"搞怪"等。
        max_iterations (int): Agent 最大迭代次数，防止无限循环。
        
    Returns:
        str: 生成的爆款文案（JSON 格式字符串）。
    """
    
    print(f"\n🚀 启动小红书文案生成助手，产品：{product_name}，风格：{tone_style}\n")
    
    # 存储对话历史，包括系统提示词和用户请求
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"请为产品「{product_name}」生成一篇小红书爆款文案。要求：语气{tone_style}，包含标题、正文、至少5个相关标签和5个表情符号。请以完整的JSON格式输出，并确保JSON内容用markdown代码块包裹（例如：```json{{...}}```）。"}
    ]
    
    iteration_count = 0
    final_response = None
    
    while iteration_count < max_iterations:
        iteration_count += 1
        print(f"-- Iteration {iteration_count} --")
        
        try:
            # 调用 DeepSeek API，传入对话历史和工具定义
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=messages,
                tools=TOOLS_DEFINITION, # 告知模型可用的工具
                tool_choice="auto" # 允许模型自动决定是否使用工具
            )

            response_message = response.choices[0].message
            
            # **ReAct模式：处理工具调用**
            if response_message.tool_calls: # 如果模型决定调用工具
                print("Agent: 决定调用工具...")
                messages.append(response_message) # 将工具调用信息添加到对话历史
                
                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    # 确保参数是合法的JSON字符串，即使工具不要求参数，也需要传递空字典
                    function_args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}

                    print(f"Agent Action: 调用工具 '{function_name}'，参数：{function_args}")
                    
                    # 查找并执行对应的模拟工具函数
                    if function_name in available_tools:
                        tool_function = available_tools[function_name]
                        tool_result = tool_function(**function_args)
                        print(f"Observation: 工具返回结果：{tool_result}")
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": str(tool_result) # 工具结果作为字符串返回
                        })
                    else:
                        error_message = f"错误：未知的工具 '{function_name}'"
                        print(error_message)
                        tool_outputs.append({
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "content": error_message
                        })
                messages.extend(tool_outputs) # 将工具执行结果作为 Observation 添加到对话历史
                
            # **ReAct 模式：处理最终内容**
            elif response_message.content: # 如果模型直接返回内容（通常是最终答案）
                print(f"[模型生成结果] {response_message.content}")
                
                # --- START: 添加 JSON 提取和解析逻辑 ---
                json_string_match = re.search(r"```json\s*(\{.*\})\s*```", response_message.content, re.DOTALL)
                
                if json_string_match:
                    extracted_json_content = json_string_match.group(1)
                    try:
                        final_response = json.loads(extracted_json_content)
                        print("Agent: 任务完成，成功解析最终JSON文案。")
                        return json.dumps(final_response, ensure_ascii=False, indent=2)
                    except json.JSONDecodeError as e:
                        print(f"Agent: 提取到JSON块但解析失败: {e}")
                        print(f"尝试解析的字符串:\n{extracted_json_content}")
                        messages.append(response_message) # 解析失败，继续对话
                else:
                    # 如果没有匹配到 ```json 块，尝试直接解析整个 content
                    try:
                        final_response = json.loads(response_message.content)
                        print("Agent: 任务完成，直接解析最终JSON文案。")
                        return json.dumps(final_response, ensure_ascii=False, indent=2)
                    except json.JSONDecodeError:
                        print("Agent: 生成了非JSON格式内容或非Markdown JSON块，可能还在思考或出错。")
                        messages.append(response_message) # 非JSON格式，继续对话
                # --- END: 添加 JSON 提取和解析逻辑 ---
            else:
                print("Agent: 未知响应，可能需要更多交互。")
                break
                
        except Exception as e:
            print(f"调用 DeepSeek API 时发生错误: {e}")
            break
    
    print("\n⚠️ Agent 达到最大迭代次数或未能生成最终文案。请检查Prompt或增加迭代次数。")
    return "未能成功生成文案。"

## 5. 实际测试与文案生成

现在，让我们调用我们构建的 `generate_rednote` 函数，看看它能生成什么样的爆款文案！

In [6]:
# 测试案例 1: 精确匹配 - 兰蔻小黑瓶精华
product_name_1 = "兰蔻小黑瓶精华"
tone_style_1 = "活泼甜美"
result_1 = generate_rednote(product_name_1, tone_style_1)

print("\n--- 生成的文案 1 ---")
print(result_1)


🚀 启动小红书文案生成助手，产品：兰蔻小黑瓶精华，风格：活泼甜美

-- Iteration 1 --
Agent: 决定调用工具...
Agent Action: 调用工具 'search_web'，参数：{'query': '兰蔻小黑瓶精华 2024 2025 最新卖点 功效'}
[Tool Call] 搜索网页：兰蔻小黑瓶精华 2024 2025 最新卖点 功效
Observation: 工具返回结果：未找到关于 '兰蔻小黑瓶精华 2024 2025 最新卖点 功效' 的特定信息，但市场反馈通常关注产品成分、功效和用户体验。
Agent Action: 调用工具 'search_web'，参数：{'query': '兰蔻小黑瓶精华 小红书 爆款文案 用户评价'}
[Tool Call] 搜索网页：兰蔻小黑瓶精华 小红书 爆款文案 用户评价
Observation: 工具返回结果：未找到关于 '兰蔻小黑瓶精华 小红书 爆款文案 用户评价' 的特定信息，但市场反馈通常关注产品成分、功效和用户体验。
Agent Action: 调用工具 'query_product_database'，参数：{'product_name': '兰蔻小黑瓶精华'}
[Tool Call] 查询产品数据库：兰蔻小黑瓶精华
Observation: 工具返回结果：产品名称：兰蔻小黑瓶精华
品牌：兰蔻 (Lancôme)
分类：精华液
价格：¥760/30ml
核心成分：二裂酵母发酵产物溶胞物、酵母提取物、腺苷、维生素E
主要功效：修护肌肤屏障、改善肌肤粗糙、提升肌肤光泽度、强韧肌底
适用人群：所有肤质，尤其适合肌肤粗糙、暗沉、初老肌
质地：蛋清质地，轻盈好吸收，不粘腻
使用方法：洁面爽肤后，取适量均匀涂抹于面部及颈部，轻轻按摩至吸收
核心卖点：全球畅销修护精华，二裂酵母精研配方，微生态护肤理念，修护强韧肌底
规格：30ml / 50ml / 75ml
评分：4.7/5.0
标签：修护精华、抗初老、二裂酵母、肌底液
-- Iteration 2 --
Agent: 决定调用工具...
Agent Action: 调用工具 'search_web'，参数：{'query': '兰蔻小黑瓶 小红书 热门笔记 2024 2025'}
[Tool Call] 搜索网页：兰蔻小黑瓶 小红书 热门笔

In [7]:
# 测试案例 2: 模糊搜索 - 美白精华
product_name_2 = "美白精华"
tone_style_2 = "知性温柔"
result_2 = generate_rednote(product_name_2, tone_style_2)

print("\n--- 生成的文案 2 ---")
print(result_2)


🚀 启动小红书文案生成助手，产品：美白精华，风格：知性温柔

-- Iteration 1 --
Agent: 决定调用工具...
Agent Action: 调用工具 'search_web'，参数：{'query': '2025小红书美白精华 爆款推荐 热门趋势'}
[Tool Call] 搜索网页：2025小红书美白精华 爆款推荐 热门趋势
Observation: 工具返回结果：小红书美白精华热门话题：冷白皮养成、烟酰胺精华测评、早C晚A搭配。用户关注：温和不刺激、淡痘印效果。
Agent Action: 调用工具 'search_web'，参数：{'query': '美白精华 用户评价 真实使用感受'}
[Tool Call] 搜索网页：美白精华 用户评价 真实使用感受
Observation: 工具返回结果：小红书美白精华热门话题：冷白皮养成、烟酰胺精华测评、早C晚A搭配。用户关注：温和不刺激、淡痘印效果。
Agent Action: 调用工具 'query_product_database'，参数：{'product_name': '美白精华'}
[Tool Call] 查询产品数据库：美白精华
Observation: 工具返回结果：找到 11 个相关产品：

产品名称：Olay小白瓶精华
品牌：Olay
分类：精华液
价格：¥259/40ml
核心成分：5%烟酰胺、十一碳烯酰基苯丙氨酸、Sepitonic、透明质酸
主要功效：美白提亮、淡化痘印、均匀肤色、改善暗沉
适用人群：非敏感肌，有美白、提亮需求者
质地：水润精华质地，流动性好
使用方法：早晚使用，洁面爽肤后涂抹，初期建立耐受，配合防晒效果更佳
核心卖点：5%黄金浓度烟酰胺美白精华，性价比美白天花板，宝洁集团技术
规格：40ml
评分：4.4/5.0
标签：美白精华、烟酰胺、提亮、淡痘印
---
产品名称：兰蔻小黑瓶精华
品牌：兰蔻 (Lancôme)
分类：精华液
价格：¥760/30ml
核心成分：二裂酵母发酵产物溶胞物、酵母提取物、腺苷、维生素E
主要功效：修护肌肤屏障、改善肌肤粗糙、提升肌肤光泽度、强韧肌底
适用人群：所有肤质，尤其适合肌肤粗糙、暗沉、初老肌
质地：蛋清质地，轻盈好吸收，不粘腻
使用方法：洁面爽肤后，取适量均匀涂抹于面部及颈部，轻轻按摩至吸收
核心卖点：全球畅销修护精

### 格式化 小红书文案

**格式化函数 `format_rednote_for_markdown` 的功能：**

1. 解析 JSON 字符串。
2. 提取标题、正文、标签和表情符号。
3. 将它们组合成一个易读的 Markdown 格式的文本。


**工作方式：**

1. **解析 JSON**：使用 `json.loads()` 将输入的字符串转换为 Python 字典。如果解析失败，会返回一个错误信息。
2. **提取数据**：使用 `.get()` 方法从字典中安全地提取 `title`、`body` 和 `hashtags`。使用 `.get()` 的好处是，如果某个键不存在，它会返回一个默认值（例如 `None` 或空列表），而不是抛出 `KeyError`。
3. **构建 Markdown 标题**：将 `title` 格式化为 Markdown 的二级标题 (`## Title`)。
4. **处理正文**：直接使用 `body`。由于小红书正文中的换行很重要，我们保留它们。
5. **处理 Hashtags**：将 `hashtags` 列表中的每个标签用空格连接起来，形成一行。
6. **表情符号 (Emojis)**：在小红书的实际发布中，表情符号通常已经嵌入在标题和正文中了。这个函数没有单独列出它们，因为这通常不是最终发布格式的一部分。如果需要，可以取消注释相关代码来单独显示它们。
7. **返回结果**：返回拼接好的 Markdown 字符串，并使用 `.strip()` 去除可能存在于末尾的多余空白。

In [8]:
import json

def format_rednote_for_markdown(json_string: str) -> str:
    """
    将 JSON 格式的小红书文案转换为 Markdown 格式，以便于阅读和发布。

    Args:
        json_string (str): 包含小红书文案的 JSON 字符串。
                           预计格式为 {"title": "...", "body": "...", "hashtags": [...], "emojis": [...]}

    Returns:
        str: 格式化后的 Markdown 文本。
    """
    try:
        data = json.loads(json_string)
    except json.JSONDecodeError as e:
        return f"错误：无法解析 JSON 字符串 - {e}\n原始字符串：\n{json_string}"

    title = data.get("title", "无标题")
    body = data.get("body", "")
    hashtags = data.get("hashtags", [])
    # 表情符号通常已经融入标题和正文中，这里可以选择是否单独列出
    # emojis = data.get("emojis", []) 

    # 构建 Markdown 文本
    markdown_output = f"## {title}\n\n" # 标题使用二级标题
    
    # 正文，保留换行符
    markdown_output += f"{body}\n\n"
    
    # Hashtags
    if hashtags:
        hashtag_string = " ".join(hashtags) # 小红书标签通常是空格分隔
        markdown_output += f"{hashtag_string}\n"
        
    # 如果需要，可以单独列出表情符号，但通常它们已经包含在标题和正文中
    # if emojis:
    #     emoji_string = " ".join(emojis)
    #     markdown_output += f"\n使用的表情：{emoji_string}\n"
        
    return markdown_output.strip() # 去除末尾多余的空白

In [9]:
# --- 示例使用 ---
# 假设这是 generate_rednote 函数的输出
generated_json_output = """
{
  "title": "✨ 28天逆袭冷白皮！这款美白精华让我告别暗沉痘印 🌟",
  "body": "姐妹们！我终于找到了我的本命美白精华！💖\\n\\n作为一个常年熬夜➕痘印困扰的混油皮，肤色暗沉一直是我的心头大患。直到遇见了这款美白精华，简直打开了新世界的大门！🤩\\n\\n🌟 核心成分：烟酰胺+VC衍生物，双管齐下，提亮肤色效果绝绝子！\\n💧 质地轻薄到爆炸，上脸秒吸收，完全不会黏腻，油皮姐妹放心冲！\\n🌿 用了28天，痘印肉眼可见变淡了，整张脸都透亮了起来，素颜也能打！\\n\\n使用方法也很简单：早晚洁面后，滴2-3滴在手心，轻轻按压上脸，后续再叠加保湿产品就OK啦～\\n\\n真心推荐给所有想要均匀肤色、告别暗沉的姐妹！入股不亏！💖",
  "hashtags": ["#美白精华", "#提亮肤色", "#淡化痘印", "#护肤好物", "#冷白皮"],
  "emojis": ["✨", "💖", "🤩", "💧", "🌿"]
}
"""

# 调用格式化函数
markdown_note = format_rednote_for_markdown(generated_json_output)

# 打印结果
print("--- 格式化后的小红书文案 (Markdown) ---")
print(markdown_note)

# --- 另一个例子，假设JSON解析失败 ---
invalid_json_output = "{'title': 'Test', 'body': 'This is not valid json'}" # 使用单引号，非法
markdown_error_note = format_rednote_for_markdown(invalid_json_output)
print("\n--- 格式化错误示例 ---")
print(markdown_error_note)


--- 格式化后的小红书文案 (Markdown) ---
## ✨ 28天逆袭冷白皮！这款美白精华让我告别暗沉痘印 🌟

姐妹们！我终于找到了我的本命美白精华！💖

作为一个常年熬夜➕痘印困扰的混油皮，肤色暗沉一直是我的心头大患。直到遇见了这款美白精华，简直打开了新世界的大门！🤩

🌟 核心成分：烟酰胺+VC衍生物，双管齐下，提亮肤色效果绝绝子！
💧 质地轻薄到爆炸，上脸秒吸收，完全不会黏腻，油皮姐妹放心冲！
🌿 用了28天，痘印肉眼可见变淡了，整张脸都透亮了起来，素颜也能打！

使用方法也很简单：早晚洁面后，滴2-3滴在手心，轻轻按压上脸，后续再叠加保湿产品就OK啦～

真心推荐给所有想要均匀肤色、告别暗沉的姐妹！入股不亏！💖

#美白精华 #提亮肤色 #淡化痘印 #护肤好物 #冷白皮

--- 格式化错误示例 ---
错误：无法解析 JSON 字符串 - Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
原始字符串：
{'title': 'Test', 'body': 'This is not valid json'}


In [9]:
# 调用格式化函数
markdown_note = format_rednote_for_markdown(result_1)

# 打印结果
print("--- 格式化后的小红书文案 (Markdown) ---")
print(markdown_note)

--- 格式化后的小红书文案 (Markdown) ---
## 素颜也发光✨ | 兰蔻小黑瓶用一个月，闺蜜问我做了啥项目！

姐妹们！！！我终于找到了一生推的修护精华！🥹

就是这瓶兰蔻小黑瓶精华，真的是「烂脸救星＋光泽制造机」二合一！！

💧先说质地：
蛋清一样清清爽爽的，流动性巨强
上脸咻一下就被吃进去了，完全不粘腻！
油皮干皮都给我冲，夏天用也毫无负担～

🔬核心成分有多能打：
里面可是满满的二裂酵母发酵产物溶胞物🏆
专攻修护皮肤屏障＋维稳肌底
加上腺苷＋VE，抗氧化提亮一把抓
坚持用下来，皮肤真的「稳如泰山」！

🌟我的使用感受：
✅ 第一周：皮肤不再闹脾气，泛红减少
✅ 第二周：脸颊摸起来滑溜溜，粗糙感拜拜
✅ 一个月：天呐！素颜也自带抛光感✨
连我妈都问我最近是不是皮肤变好了哈哈哈

💡怎么用效果最好？
洁面→爽肤水→小黑瓶精华（按压两泵，全脸带脖子）→乳液/面霜
早晚都用，雷打不动！

有一说一，虽然价格小贵，但一分钱一分货啊姐妹们！
30ml够用两个月左右，算下来每天也就一杯奶茶钱🥤
给自己脸蛋投资，这波血赚不亏！

真的！用过就回不去了！趁活动赶紧囤！🏃‍♀️💨

#兰蔻小黑瓶 #兰蔻精华 #修护精华推荐 #抗初老必备 #素颜好皮肤 #护肤日常 #我的护肤好物 #精华液推荐


In [10]:
# 测试案例 3: 关键词模糊匹配 - 敏感肌面霜
product_name_3 = "敏感肌面霜"
tone_style_3 = "真诚种草"
result_3 = generate_rednote(product_name_3, tone_style_3)

print("\n--- 生成的文案 3 ---")
print(result_3)


🚀 启动小红书文案生成助手，产品：敏感肌面霜，风格：真诚种草

-- Iteration 1 --
Agent: 决定调用工具...
Agent Action: 调用工具 'search_web'，参数：{'query': '敏感肌面霜 2025 热门 小红书 推荐'}
[Tool Call] 搜索网页：敏感肌面霜 2025 热门 小红书 推荐
Observation: 工具返回结果：小红书敏感肌修护热门话题：屏障修复、泛红急救、温和成分党。热门成分：神经酰胺、积雪草、泛醇。
Agent Action: 调用工具 'search_web'，参数：{'query': '敏感肌面霜 成分功效 修复屏障 保湿'}
[Tool Call] 搜索网页：敏感肌面霜 成分功效 修复屏障 保湿
Observation: 工具返回结果：小红书敏感肌修护热门话题：屏障修复、泛红急救、温和成分党。热门成分：神经酰胺、积雪草、泛醇。
Agent Action: 调用工具 'query_product_database'，参数：{'product_name': '敏感肌面霜'}
[Tool Call] 查询产品数据库：敏感肌面霜
Observation: 工具返回结果：找到 5 个相关产品：

产品名称：优色林舒安修护霜
品牌：优色林 (Eucerin)
分类：面霜
价格：¥189/50ml
核心成分：角鲨烷、SymSitive®1609、甘草查尔酮、泛醇
主要功效：舒缓泛红、修护屏障、减少敏感反应、保湿滋润
适用人群：敏感肌、泛红肌、屏障受损、不耐受肌
质地：轻盈乳霜质地，温和无刺激
使用方法：早晚护肤最后一步，取适量涂抹面部，泛红区域可加强
核心卖点：拜尔斯道夫集团敏感肌科技，SymSitive®即时舒缓泛红，德国皮肤学认证
规格：50ml
评分：4.6/5.0
标签：敏感肌面霜、舒缓泛红、屏障修护、德国科技
---
产品名称：科颜氏高保湿面霜
品牌：科颜氏 (Kiehl's)
分类：面霜
价格：¥315/50ml
核心成分：角鲨烷、冰川保护蛋白、神经酰胺、白茅根提取物
主要功效：深层保湿、锁水修护、舒缓干燥、强化肌肤屏障
适用人群：干性及中性肤质，尤其适合冬季干燥、脱皮肌
质地：丰润奶油质地，延展性好，润而不油
使用方法：早晚护肤最后一步，取适量于掌心温热后按压面部
核心卖点：经典保

## 6. 评估与优化

文案生成并非一蹴而就，需要持续的评估和优化。本节讨论一些评估方法和优化策略。

#### 评估文案质量：
*   **客观量化评估 (数据)：**
    *   **点赞/收藏/评论/分享：** 基础互动
    *   **曝光/阅读/点击/涨粉：：** 流量与曝光
    *   **停留时长/截图率：** 用户行为。
    *   **商品页浏览/加购/ROI/成交转化：** 商业价值
    *   **爆文率/同类横向对比：** 竞争对比
*   **主观内部评估 (人工)：**
    *   **相关性：** 是否符合产品特点和主题。
    *   **吸引力：** 标题是否抓人，内容是否流畅。
    *   **合规性：** 是否有敏感词、违规宣传。
    *   **风格匹配：** 是否符合小红书调性和指定语气。
    *   **用户画像：** 目标人群年龄、地域、兴趣标签。



#### 优化迭代方法：
*   **Prompt 调整：** 根据评估结果，精修 System Prompt、User Prompt，增加或修改 Few-shot 示例。
*   **工具扩充：** 引入新的工具（如敏感词检测工具、竞品分析工具）。
*   **RAG (检索增强生成)：** 结合更精准的内部知识库，减少幻觉。


## 7. 总结与展望

通过本次实战，我们成功构建了一个基于 DeepSeek Agent 的小红书爆款文案生成助手。我们学习了如何拆解需求、设计 Prompt、定义工具，并实现 Agent 的核心工作流。

Agent 在内容营销领域的潜力巨大，未来可以进一步拓展到：

*   **超个性化内容：** 根据用户数据，生成一对一的定制文案。
*   **多模态内容创作：** 结合图片、视频生成，实现图文音视频一体化。
*   **智能营销决策：** Agent 不仅生成内容，还能分析效果并给出投放建议。
*   **跨平台适配：** 快速生成适应不同社交媒体平台风格的文案。

同时，我们也需关注挑战，如确保内容真实性、处理高度主观情感、与现有工作流的无缝集成等。Agent 技术仍在快速发展，期待未来能带来更多惊喜！